<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [20]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [21]:
# Posibles bandas
os.listdir("./songs_dataset/")

['prince.txt',
 'dickinson.txt',
 'notorious-big.txt',
 'beatles.txt',
 'bob-dylan.txt',
 'bjork.txt',
 'johnny-cash.txt',
 'disney.txt',
 'janisjoplin.txt',
 'kanye.txt',
 'bob-marley.txt',
 'leonard-cohen.txt',
 'ludacris.txt',
 'adele.txt',
 'alicia-keys.txt',
 'joni-mitchell.txt',
 'amy-winehouse.txt',
 'lorde.txt',
 'rihanna.txt',
 'Kanye_West.txt',
 'nirvana.txt',
 'cake.txt',
 'bieber.txt',
 'notorious_big.txt',
 'missy-elliott.txt',
 'dolly-parton.txt',
 'jimi-hendrix.txt',
 'michael-jackson.txt',
 'al-green.txt',
 'lil-wayne.txt',
 'lady-gaga.txt',
 'lin-manuel-miranda.txt',
 'nursery_rhymes.txt',
 'dj-khaled.txt',
 'radiohead.txt',
 'patti-smith.txt',
 'blink-182.txt',
 'Lil_Wayne.txt',
 'dr-seuss.txt',
 'r-kelly.txt',
 'drake.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'nicki-minaj.txt',
 'kanye-west.txt',
 'paul-simon.txt',
 'nickelback.txt',
 'eminem.txt',
 'bruno-mars.txt']

In [22]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/var/folders/qx/hv433zj1493bs872_zqkgspm0000gn/T/ipykernel_78647/3849064916.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [23]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [24]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [25]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [26]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [27]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [28]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [29]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [30]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [31]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 113045.2421875
Loss after epoch 1: 65966.5703125
Loss after epoch 2: 65935.015625
Loss after epoch 3: 65718.296875
Loss after epoch 4: 63874.96875
Loss after epoch 5: 64160.875
Loss after epoch 6: 64080.5
Loss after epoch 7: 64815.09375
Loss after epoch 8: 62632.75
Loss after epoch 9: 60452.75
Loss after epoch 10: 59839.6875
Loss after epoch 11: 58884.25
Loss after epoch 12: 57715.9375
Loss after epoch 13: 56494.1875
Loss after epoch 14: 55817.25
Loss after epoch 15: 55843.0625
Loss after epoch 16: 51722.6875
Loss after epoch 17: 49857.875
Loss after epoch 18: 49592.125
Loss after epoch 19: 48960.0


(156986, 287740)

### 4 - Ensayar

In [32]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('pretty', 0.8954242467880249),
 ('sleep', 0.8665637969970703),
 ('help', 0.8439376950263977),
 ('cry', 0.8351294994354248),
 ('not', 0.8309615254402161),
 ('try', 0.8276941180229187),
 ('peace', 0.8144854307174683),
 ('little', 0.8140544295310974),
 ('twist', 0.8123906254768372),
 ('seems', 0.8079560995101929)]

In [33]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('shake', -0.22872784733772278),
 ('four', -0.23302021622657776),
 ('five', -0.237460196018219),
 ('six', -0.2378387749195099),
 ('bang', -0.2483242005109787),
 ('our', -0.25538894534111023),
 ('day', -0.2689763307571411),
 ('going', -0.2692073881626129),
 ('here', -0.26990994811058044),
 ('three', -0.28389739990234375)]

In [34]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9813730716705322),
 ('three', 0.9745771884918213),
 ('six', 0.9710819721221924),
 ('seven', 0.9584385752677917),
 ('two', 0.9517235159873962),
 ('sixty', 0.8990404605865479),
 ('one', 0.7951189279556274),
 ('crying', 0.7946323752403259),
 ('us', 0.774010956287384),
 ("i'm", 0.7508407235145569)]

In [35]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[("can't", 0.9434012770652771),
 ('buy', 0.9396973252296448),
 ('much', 0.9033190011978149),
 ('just', 0.8509107232093811),
 ('hide', 0.8355359435081482)]

In [36]:
# Ensayar con una palabra que no está en el vocabulario:
try:
    w2v_model.wv.most_similar(negative=["diedaa"])
except Exception as e:
    print("Error:", e)

Error: "Key 'diedaa' not present in vocabulary"


In [37]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 0.06138294  0.05881053 -0.06370477  0.02445546 -0.20152311 -0.18612298
 -0.15284492  0.4548641  -0.042179    0.03535716  0.13657664 -0.1851985
 -0.18126567  0.22149132 -0.3038017  -0.23970504  0.0709474  -0.05679084
 -0.05165781 -0.23843321 -0.08529595  0.19564523 -0.07678778  0.03796823
  0.07516409 -0.04826145  0.07379413  0.10396487  0.00737745 -0.22765112
 -0.04566921  0.1293697   0.27785546  0.19387001 -0.13509622  0.20857269
  0.4091703  -0.00386562 -0.10631571 -0.09057102  0.02400816 -0.08005267
  0.13399461  0.08833719 -0.01894974  0.08592648 -0.15905555  0.10259393
  0.1445954  -0.1209247  -0.279195   -0.04061342  0.11381969  0.31366253
 -0.07408687  0.1397688   0.22791368  0.1320968  -0.01811604  0.09772383
  0.09249984 -0.14871608 -0.16348287 -0.13202605 -0.0983391   0.02714311
  0.16531329  0.2605218  -0.03259389 -0.02894598  0.11621305 -0.06974123
  0.09562718 -0.1527624   0.22071022  0.15996738  0.15890466 -0.04710886
 -0.12556134 -0.03993148 -0.10795182  0.01878742  0.

In [38]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0000001192092896),
 ('babe', 0.9085147380828857),
 ('someone', 0.8886134028434753),
 ('need', 0.8827983736991882),
 ('nothing', 0.8740261793136597),
 ("didn't", 0.8638365268707275),
 ("there's", 0.8526694774627686),
 ('you', 0.8456733226776123),
 ('feed', 0.844502866268158),
 ('somebody', 0.8362798094749451)]

In [39]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9085147380828857),
 ('someone', 0.8886134028434753),
 ('need', 0.8827983736991882),
 ('nothing', 0.8740261793136597),
 ("didn't", 0.8638364672660828),
 ("there's", 0.8526694774627686),
 ('you', 0.8456733226776123),
 ('feed', 0.844502866268158),
 ('somebody', 0.8362798094749451),
 ('buy', 0.8351726531982422)]

### 5 - Visualizar agrupación de vectores

In [40]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [41]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

In [42]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [43]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs.
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

### Resolución — Bob Dylan

Se elige Bob Dylan por su extensa discografía con vocabulario variado y temáticas recurrentes (tiempo, viaje, amor, protesta social).

In [44]:
df_dylan = pd.read_csv('songs_dataset/bob-dylan.txt', sep='/n', header=None, engine='python')
print('Cantidad de documentos:', df_dylan.shape[0])
df_dylan.head()

Cantidad de documentos: 5213


,0
0,"“There must be some way out of here,” said the..."
1,"“There’s too much confusion, I can’t get no re..."
2,"Businessmen, they drink my wine, plowmen dig m..."
3,None of them along the line know what any of i...
4,"“No reason to get excited,” the thief, he kind..."


In [45]:
dylan_tokens = []
for _, row in df_dylan.iterrows():
    dylan_tokens.append(text_to_word_sequence(row[0]))
dylan_tokens[:2]

[['“there',
  'must',
  'be',
  'some',
  'way',
  'out',
  'of',
  'here',
  '”',
  'said',
  'the',
  'joker',
  'to',
  'the',
  'thief'],
 ['“there’s', 'too', 'much', 'confusion', 'i', 'can’t', 'get', 'no', 'relief']]

In [46]:
dylan_model = Word2Vec(
    min_count=5,
    window=2,
    vector_size=300,
    negative=20,
    workers=multiprocessing.cpu_count(),
    sg=1
)
dylan_model.build_vocab(dylan_tokens)
print('Cantidad de docs en el corpus:', dylan_model.corpus_count)
print('Cantidad de words distintas en el corpus:', len(dylan_model.wv.index_to_key))

Cantidad de docs en el corpus: 5213
Cantidad de words distintas en el corpus: 948


In [47]:
dylan_model.train(
    dylan_tokens,
    total_examples=dylan_model.corpus_count,
    epochs=20,
    compute_loss=True,
    callbacks=[callback()]
)

Loss after epoch 0: 71798.6484375
Loss after epoch 1: 48590.3046875
Loss after epoch 2: 48106.515625
Loss after epoch 3: 49020.015625
Loss after epoch 4: 46693.921875
Loss after epoch 5: 47882.46875
Loss after epoch 6: 47646.8125
Loss after epoch 7: 47308.71875
Loss after epoch 8: 47403.53125
Loss after epoch 9: 45476.4375
Loss after epoch 10: 42290.8125
Loss after epoch 11: 41770.5625
Loss after epoch 12: 43536.5625
Loss after epoch 13: 43945.0
Loss after epoch 14: 43402.125
Loss after epoch 15: 43031.75
Loss after epoch 16: 42246.375
Loss after epoch 17: 35863.4375
Loss after epoch 18: 40889.25
Loss after epoch 19: 35409.6875


(443395, 768300)

### Términos de interés

In [48]:
# Palabras que MÁS se relacionan con 'time'
dylan_model.wv.most_similar(positive=['time'], topn=10)

[('ago', 0.7836059927940369),
 ('day', 0.739132285118103),
 ('either', 0.7184117436408997),
 ('easy', 0.7119137644767761),
 ('such', 0.7071001529693604),
 ('hair', 0.6980146765708923),
 ('coat', 0.6966164112091064),
 ('stuff', 0.6894205212593079),
 ('worth', 0.6875837445259094),
 ('isn’t', 0.6869775652885437)]

**'time'** aparece asociado a palabras como 'ago', 'day', 'worth' y 'either'. El modelo captura el uso temporal ('ago', 'day') pero también agrupa palabras que aparecen en construcciones estructuralmente similares ('such', 'easy', 'worth'). La presencia de 'hair' y 'coat' sugiere que en el corpus estas palabras co-ocurren en posiciones sintácticas parecidas a 'time', lo que muestra una limitación del modelo dado el tamaño reducido del vocabulario.

In [49]:
# Palabras que MÁS se relacionan con 'love'
dylan_model.wv.most_similar(positive=['love'], topn=10)

[('shot', 0.757122278213501),
 ('help', 0.737695038318634),
 ('sick', 0.706072986125946),
 ('need', 0.702987015247345),
 ('isn’t', 0.6899672150611877),
 ('doctor', 0.685260534286499),
 ('breath', 0.683557391166687),
 ('life', 0.6808125972747803),
 ('kind', 0.6762580871582031),
 ('first', 0.6747739911079407)]

**'love'** aparece asociado a palabras de vulnerabilidad y necesidad: 'shot', 'sick', 'need', 'help', 'doctor', 'breath'. En Dylan el amor no es idealizado sino que se expresa como sufrimiento y desesperación emocional, consistente con el tono crudo de álbumes como *Blood on the Tracks*.

In [50]:
# Palabras que MENOS se relacionan con 'love'
dylan_model.wv.most_similar(negative=['love'], topn=10)

[('are', -0.14974814653396606),
 ('at', -0.1664968580007553),
 ('high', -0.17038507759571075),
 ('from', -0.19170168042182922),
 ('they', -0.1975751370191574),
 ('around', -0.20042544603347778),
 ('down', -0.20134171843528748),
 ('there', -0.20232845842838287),
 ('let', -0.2046290636062622),
 ('on', -0.20914117991924286)]

Los resultados negativos de **'love'** son principalmente palabras funcionales: preposiciones, pronombres y auxiliares ('are', 'at', 'from', 'they', 'on'). Esto indica que el contexto sintáctico de 'love' es muy distinto al de estas palabras gramaticales; no existe un antónimo semántico claro dentro del vocabulario de Dylan.

In [51]:
# Palabras que MÁS se relacionan con 'rain'
dylan_model.wv.most_similar(positive=['rain'], topn=10)

[('land', 0.9347125887870789),
 ('dawn', 0.9321924448013306),
 ('stars', 0.9314847588539124),
 ('flowers', 0.9310208559036255),
 ('wild', 0.9272540807723999),
 ('shadows', 0.9269667863845825),
 ('earth', 0.9260287880897522),
 ('mother', 0.9258796572685242),
 ('’neath', 0.9245126843452454),
 ('unto', 0.9242629408836365)]

**'rain'** aparece fuertemente asociado a imágenes poéticas de la naturaleza: 'land', 'dawn', 'stars', 'flowers', 'earth', 'shadows'. Los scores son muy altos (≈0.93), lo que indica que estas palabras aparecen en contextos líricamente muy similares. Dylan las usa juntas en canciones de imagería folk-bíblica como *A Hard Rain's A-Gonna Fall*, donde la lluvia y la tierra son metáforas de devastación y esperanza.

### Reducción de dimensionalidad y visualización

In [52]:
vecs_dylan, labels_dylan = reduce_dimensions(dylan_model)

MAX_WORDS = 100
fig = px.scatter(
    x=vecs_dylan[:MAX_WORDS, 0],
    y=vecs_dylan[:MAX_WORDS, 1],
    text=labels_dylan[:MAX_WORDS]
)
fig.show()

Al inspeccionar el gráfico se identifican al menos tres grupos:

- **Imágenes de la naturaleza**: *rain*, *land*, *dawn*, *earth*, *stars*, *flowers* agrupados. Reflejan el uso de imagería natural en canciones de tono folk-bíblico.
- **Vulnerabilidad y necesidad**: *love*, *shot*, *sick*, *need*, *help*, *doctor* próximos entre sí. El modelo captura el registro emocional crudo con que Dylan aborda el amor y el sufrimiento.
- **Palabras funcionales**: preposiciones, pronombres y auxiliares tienden a agruparse separados del vocabulario temático, lo que es esperable ya que aparecen en distribuciones sintácticas uniformes.